In [6]:
!pip install tokenizers datasets pandas -q

import shutil, glob, os, torch, gc
from config import ModelConfig, TrainConfig
from train import train

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

all_summaries = {}

Device: cuda


In [7]:
cleanup()
model, summary = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="sinusoidal"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="baseline"
)
all_summaries["baseline"] = summary
del model


Experiment: baseline
Device: cuda



README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training BPE tokenizer...



Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 2579
  Val chunks:   272
  Test chunks:  312
Model: standard attn | sinusoidal PE | conv=None
  Parameters: 4,668,928

Epoch 1/15 | Train Loss 9.0360 PPL 8399.89 | Val Loss 8.5156 PPL 4992.08 | Throughput 52935 tok/s | Time 49.9s

  ✓ Saved best checkpoint (PPL 4992.08)
  Step    100 | Loss 8.3987 | PPL 4441.38 | LR 6.00e-05 | Mem 11422MB

Epoch 2/15 | Train Loss 7.9733 PPL 2902.51 | Val Loss 7.4522 PPL 1723.69 | Throughput 52012 tok/s | Time 50.8s

  ✓ Saved best checkpoint (PPL 1723.69)
  Step    200 | Loss 7.3503 | PPL 1556.60 | LR 1.20e-04 | Mem 11422MB

Epoch 3/15 | Train Loss 7.3053 PPL 1488.14 | Val Loss 7.2468 PPL 1403.64 | Throughput 49006 tok/s | Time 53.9s

  ✓ Saved best checkpoint (PPL 1403.64)
  Step    300 | Loss 7.1919 | PPL 1328.56 | LR 1.80e-04 | Mem 11422MB

Epoch 4/15 | Train Loss 7.1486 PPL 1272.36 | Val Loss 6.9670 PPL 1061.06

In [8]:

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="sliding_window", pos_enc_type="sinusoidal", window_size=256),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="attn_sliding_window"
)
all_summaries["sliding_window"] = s

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="linear", pos_enc_type="sinusoidal"),
    TrainConfig(context_len=1024, batch_size=16, epochs=15, lr=3e-4),
    experiment_name="attn_linear"
)
all_summaries["linear"] = s


cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="gqa", pos_enc_type="sinusoidal", n_kv_heads=2),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="attn_gqa"
)
all_summaries["gqa"] = s


Experiment: attn_sliding_window
Device: cuda

Loading existing tokenizer from tokenizer.json
Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 2579
  Val chunks:   272
  Test chunks:  312
Model: sliding_window attn | sinusoidal PE | conv=None
  Parameters: 4,668,928

Epoch 1/15 | Train Loss 9.0399 PPL 8433.14 | Val Loss 8.5187 PPL 5007.61 | Throughput 48905 tok/s | Time 54.0s

  ✓ Saved best checkpoint (PPL 5007.61)
  Step    100 | Loss 8.4011 | PPL 4451.99 | LR 6.00e-05 | Mem 11425MB

Epoch 2/15 | Train Loss 7.9736 PPL 2903.34 | Val Loss 7.4500 PPL 1719.82 | Throughput 49720 tok/s | Time 53.1s

  ✓ Saved best checkpoint (PPL 1719.82)
  Step    200 | Loss 7.3485 | PPL 1553.85 | LR 1.20e-04 | Mem 11425MB

Epoch 3/15 | Train Loss 7.3038 PPL 1485.91 | Val Loss 7.2444 PPL 1400.20 | Throughput 49615 tok/s | Time 53.2s

  ✓ Saved best checkpoint (PPL 1400.20)
  Step    300 | Loss 7.1820 | PPL 1315.50 | LR 1.80e-04 | Mem 11425MB

E

In [9]:
%%writefile /kaggle/working/evaluate.py

import json
import math
import time
from pathlib import Path
from typing import Dict, List

import torch
import torch.nn as nn
import pandas as pd

from config import ModelConfig, TrainConfig
from model import build_model
from data import get_dataloaders
from utils import get_device, set_seed


@torch.no_grad()
def compute_perplexity(model, dataloader, device) -> dict:
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss += loss.item()
        total_tokens += y.numel()
    avg_loss = total_loss / total_tokens
    ppl = math.exp(min(avg_loss, 100))
    return {"loss": avg_loss, "perplexity": ppl}


@torch.no_grad()
def measure_throughput(model, seq_len, batch_size=8, vocab_size=10000, n_runs=10, device=None):
    if device is None:
        device = get_device()
    model.eval().to(device)
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    for _ in range(3):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    total_tokens = batch_size * seq_len * n_runs
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return total_tokens / elapsed


@torch.no_grad()
def measure_peak_memory(model, seq_len, batch_size=8, vocab_size=10000, device=None):
    if device is None:
        device = get_device()
    if not torch.cuda.is_available():
        return 0.0
    model.eval().to(device)
    torch.cuda.reset_peak_memory_stats()
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    _ = model(x)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


def benchmark_model(model_config, context_lengths, batch_size=8):
    device = get_device()
    results = []
    for seq_len in context_lengths:
        print(f"  Benchmarking seq_len={seq_len}")
        model = build_model(model_config).to(device)
        throughput = measure_throughput(model, seq_len, batch_size, model_config.vocab_size, device=device)
        memory = measure_peak_memory(model, seq_len, batch_size, model_config.vocab_size, device=device)
        results.append({
            "attention": model_config.attention_type,
            "pos_enc": model_config.pos_enc_type,
            "conv_hybrid": model_config.conv_hybrid or "none",
            "seq_len": seq_len,
            "n_params": model.count_parameters(),
            "throughput_toks": throughput,
            "peak_mem_mb": memory,
        })
        print(f"    Throughput: {throughput:.0f} tok/s | Memory: {memory:.1f} MB")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results


def extrapolation_test(model_config, checkpoint_path, train_len=512, test_lens=None, batch_size=16):
    if test_lens is None:
        test_lens = [512, 1024, 2048]
    device = get_device()
    results = []
    for test_len in test_lens:
        print(f"  Extrapolation test: seq_len={test_len}")
        _, val_loader, _, tokenizer = get_dataloaders(
            context_len=test_len, batch_size=batch_size, vocab_size=model_config.vocab_size,
        )
        model_config.vocab_size = tokenizer.get_vocab_size()
        model = build_model(model_config).to(device)
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        metrics = compute_perplexity(model, val_loader, device)
        metrics["seq_len"] = test_len
        results.append(metrics)
        print(f"    PPL: {metrics['perplexity']:.2f}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results


def generate_comparison_table(results_dir="logs", output_path="results_comparison.csv"):
    results_dir = Path(results_dir)
    rows = []
    for summary_file in results_dir.glob("*_summary.json"):
        with open(summary_file) as f:
            rows.append(json.load(f))
    if rows:
        df = pd.DataFrame(rows)
        df = df.sort_values("best_val_ppl")
        df.to_csv(output_path, index=False)
        print(f"\nComparison table saved to {output_path}")
        print(df.to_string(index=False))
        return df
    else:
        print("No experiment summaries found.")
        return None

Overwriting /kaggle/working/evaluate.py


In [10]:
cleanup()
from evaluate import benchmark_model

attn_types = ["standard", "sliding_window", "linear", "gqa"]
context_lengths = [512, 1024, 2048]
bench_results = []

for attn in attn_types:
    cleanup()
    cfg = ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                      attention_type=attn, pos_enc_type="sinusoidal",
                      window_size=256, n_kv_heads=2)
    results = benchmark_model(cfg, context_lengths, batch_size=8)
    bench_results.extend(results)

import pandas as pd
df_bench = pd.DataFrame(bench_results)
df_bench.to_csv("attention_benchmarks.csv", index=False)
print(df_bench.to_string(index=False))

  Benchmarking seq_len=512
Model: standard attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 195181 tok/s | Memory: 237.2 MB
  Benchmarking seq_len=1024
Model: standard attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 149949 tok/s | Memory: 398.2 MB
  Benchmarking seq_len=2048
Model: standard attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 98506 tok/s | Memory: 1184.8 MB
  Benchmarking seq_len=512
Model: sliding_window attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 195432 tok/s | Memory: 237.2 MB
  Benchmarking seq_len=1024
Model: sliding_window attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 150105 tok/s | Memory: 398.2 MB
  Benchmarking seq_len=2048
Model: sliding_window attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
    Throughput: 99178 tok/s | Memory: 1188.8 MB
  Benchmarking seq_len=512
Model: linear attn | sinusoidal PE | conv=None
  Parameters: 4,

In [11]:
cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="rope"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="pe_rope"
)
all_summaries["rope"] = s

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="alibi"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="pe_alibi"
)
all_summaries["alibi"] = s

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="relative"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="pe_relative"
)
all_summaries["relative"] = s


Experiment: pe_rope
Device: cuda

Loading existing tokenizer from tokenizer.json
Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 2579
  Val chunks:   272
  Test chunks:  312
Model: standard attn | rope PE | conv=None
  Parameters: 4,668,928

Epoch 1/15 | Train Loss 8.9888 PPL 8012.44 | Val Loss 8.4522 PPL 4685.16 | Throughput 46800 tok/s | Time 56.4s

  ✓ Saved best checkpoint (PPL 4685.16)
  Step    100 | Loss 8.3464 | PPL 4214.93 | LR 6.00e-05 | Mem 11458MB

Epoch 2/15 | Train Loss 7.9327 PPL 2787.05 | Val Loss 7.3907 PPL 1620.81 | Throughput 47320 tok/s | Time 55.8s

  ✓ Saved best checkpoint (PPL 1620.81)
  Step    200 | Loss 7.2636 | PPL 1427.38 | LR 1.20e-04 | Mem 11458MB

Epoch 3/15 | Train Loss 7.1440 PPL 1266.47 | Val Loss 6.9066 PPL 998.80 | Throughput 47205 tok/s | Time 55.9s

  ✓ Saved best checkpoint (PPL 998.80)
  Step    300 | Loss 6.7482 | PPL 852.48 | LR 1.80e-04 | Mem 11458MB

Epoch 4/15 | Train Loss 6.68

In [12]:
from evaluate import extrapolation_test

pe_variants = ["sinusoidal", "rope", "alibi", "relative"]


for pe in pe_variants:
    cleanup()
    _, s = train(
        ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                    attention_type="standard", pos_enc_type=pe),
        TrainConfig(context_len=512, batch_size=32, epochs=15, lr=3e-4),
        experiment_name=f"extrap_{pe}"
    )


extrap_results = {}
for pe in pe_variants:
    cleanup()
    cfg = ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                      attention_type="standard", pos_enc_type=pe)
    results = extrapolation_test(
        model_config=cfg,
        checkpoint_path=f"checkpoints/extrap_{pe}_best.pt",
        train_len=512,
        test_lens=[512, 1024, 2048],
        batch_size=8,
    )
    extrap_results[pe] = results


print(f"\n{'PE Type':<15} {'L=512':>10} {'L=1024':>10} {'L=2048':>10}")
print("-" * 50)
for pe, results in extrap_results.items():
    ppls = {r['seq_len']: r['perplexity'] for r in results}
    print(f"{pe:<15} {ppls[512]:>10.2f} {ppls[1024]:>10.2f} {ppls[2048]:>10.2f}")


Experiment: extrap_sinusoidal
Device: cuda

Loading existing tokenizer from tokenizer.json
Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 5154
  Val chunks:   544
  Test chunks:  624
Model: standard attn | sinusoidal PE | conv=None
  Parameters: 4,668,928
  Step    100 | Loss 8.9215 | PPL 7491.22 | LR 6.00e-05 | Mem 4636MB

Epoch 1/15 | Train Loss 8.5367 PPL 5098.49 | Val Loss 7.5026 PPL 1812.84 | Throughput 64263 tok/s | Time 41.1s

  ✓ Saved best checkpoint (PPL 1812.84)
  Step    200 | Loss 7.3862 | PPL 1613.50 | LR 1.20e-04 | Mem 4637MB
  Step    300 | Loss 7.2815 | PPL 1453.15 | LR 1.80e-04 | Mem 4636MB

Epoch 2/15 | Train Loss 7.2537 PPL 1413.39 | Val Loss 7.0145 PPL 1112.64 | Throughput 65617 tok/s | Time 40.2s

  ✓ Saved best checkpoint (PPL 1112.64)
  Step    400 | Loss 6.8612 | PPL 954.54 | LR 2.40e-04 | Mem 4637MB

Epoch 3/15 | Train Loss 6.6710 PPL 789.18 | Val Loss 6.3111 PPL 550.63 | Throughput 64769 tok/s |

In [13]:
cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="sinusoidal", conv_hybrid="conv_before"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="conv_before"
)
all_summaries["conv_before"] = s

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="sinusoidal", conv_hybrid="interleaved"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="conv_interleaved"
)
all_summaries["conv_interleaved"] = s

cleanup()
_, s = train(
    ModelConfig(d_model=256, n_heads=4, n_layers=4, d_ff=512,
                attention_type="standard", pos_enc_type="sinusoidal", conv_hybrid="gated_conv_ffn"),
    TrainConfig(context_len=1024, batch_size=32, epochs=15, lr=3e-4),
    experiment_name="conv_gated_ffn"
)
all_summaries["conv_gated_ffn"] = s


Experiment: conv_before
Device: cuda

Loading existing tokenizer from tokenizer.json
Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 2579
  Val chunks:   272
  Test chunks:  312
Model: standard attn | sinusoidal PE | conv=conv_before
  Parameters: 5,982,720

Epoch 1/15 | Train Loss 8.9526 PPL 7728.18 | Val Loss 8.4547 PPL 4697.28 | Throughput 41091 tok/s | Time 64.3s

  ✓ Saved best checkpoint (PPL 4697.28)
  Step    100 | Loss 8.3619 | PPL 4280.88 | LR 6.00e-05 | Mem 11893MB

Epoch 2/15 | Train Loss 7.9748 PPL 2906.83 | Val Loss 7.4772 PPL 1767.33 | Throughput 42101 tok/s | Time 62.7s

  ✓ Saved best checkpoint (PPL 1767.33)
  Step    200 | Loss 7.3717 | PPL 1590.35 | LR 1.20e-04 | Mem 11893MB

Epoch 3/15 | Train Loss 7.3133 PPL 1500.09 | Val Loss 7.2147 PPL 1359.24 | Throughput 42230 tok/s | Time 62.5s

  ✓ Saved best checkpoint (PPL 1359.24)
  Step    300 | Loss 7.0915 | PPL 1201.66 | LR 1.80e-04 | Mem 11893MB

Epoch 4/

In [14]:
import pandas as pd
import json

# Print final comparison
print(f"\n{'='*80}")
print(f"{'Experiment':<25} {'Test PPL':>10} {'Val PPL':>10} {'Params':>10} {'Attention':<15} {'PE':<12} {'Conv':<15}")
print(f"{'='*80}")
for name, s in all_summaries.items():
    print(f"{s['experiment']:<25} {s['test_ppl']:>10.2f} {s['best_val_ppl']:>10.2f} {s['n_params']:>10,} {s['attention_type']:<15} {s['pos_enc_type']:<12} {str(s.get('conv_hybrid','none')):<15}")

with open("all_summaries.json", "w") as f:
    json.dump(list(all_summaries.values()), f, indent=2)

from evaluate import generate_comparison_table
generate_comparison_table()


Experiment                  Test PPL    Val PPL     Params Attention       PE           Conv           
baseline                      267.19     269.65  4,668,928 standard        sinusoidal   None           
attn_sliding_window           263.71     266.34  4,668,928 sliding_window  sinusoidal   None           
attn_linear                   202.13     205.41  4,668,928 linear          sinusoidal   None           
attn_gqa                      274.17     277.55  4,405,760 gqa             sinusoidal   None           
pe_rope                       175.85     177.82  4,668,928 standard        rope         None           
pe_alibi                      176.45     178.59  4,668,928 standard        alibi        None           
pe_relative                   224.36     228.18  4,669,956 standard        relative     None           
conv_before                   209.56     209.40  5,982,720 standard        sinusoidal   conv_before    
conv_interleaved              215.46     215.61  6,238,720 stan

,experiment,best_val_ppl,test_ppl,test_loss,n_params,attention_type,pos_enc_type,conv_hybrid,context_len
5,extrap_rope,131.389582,131.826980,4.881490,4668928,standard,rope,None,512
13,extrap_alibi,135.989043,135.283172,4.907370,4668928,standard,alibi,None,512
8,extrap_relative,175.423832,172.743914,5.151810,4669956,standard,relative,None,512
4,pe_rope,177.822683,175.846657,5.169612,4668928,standard,rope,None,1024
0,extrap_sinusoidal,178.504811,176.108634,5.171101,4668928,standard,sinusoidal,None,512
1,pe_alibi,178.590340,176.450070,5.173038,4668928,standard,alibi,None,1024
10,attn_linear,205.405256,202.130949,5.308916,4668928,linear,sinusoidal,None,1024
3,conv_gated_ffn,205.757122,205.138403,5.323685,9391616,standard,sinusoidal,gated_conv_ffn,1024
11,conv_before,209.400941,209.564792,5.345033,5982720,standard,sinusoidal,conv_before,1024
9,conv_interleaved,215.611291,215.461406,5.372782,6238720,standard,sinusoidal,interleaved,1024


In [17]:
import json, os
import pandas as pd
from pathlib import Path

log_dir = Path("logs")

report_rows = []

for summary_file in sorted(log_dir.glob("*_summary.json")):
    exp_name = summary_file.stem.replace("_summary", "")
    
    with open(summary_file) as f:
        summary = json.load(f)
    
    
    log_file = log_dir / f"{exp_name}_log.jsonl"
    epoch_entries = []
    if log_file.exists():
        with open(log_file) as f:
            for line in f:
                entry = json.loads(line)
                if "epoch_time_s" in entry:
                    epoch_entries.append(entry)
    
    row = {
        "Experiment": exp_name,
        "Attention": summary.get("attention_type", ""),
        "Pos Enc": summary.get("pos_enc_type", ""),
        "Conv": summary.get("conv_hybrid", "none") or "none",
        "Ctx Len": summary.get("context_len", ""),
        "Params": summary.get("n_params", ""),
        "Test PPL": round(summary.get("test_ppl", 0), 2),
        "Val PPL": round(summary.get("best_val_ppl", 0), 2),
        "Test Loss": round(summary.get("test_loss", 0), 4),
    }
    
    if epoch_entries:
        row["Train Loss"] = round(epoch_entries[-1].get("train_loss", 0), 4)
        row["Val Loss"] = round(epoch_entries[-1].get("val_loss", 0), 4)
        row["Peak Mem (MB)"] = round(max(e.get("peak_mem_mb", 0) for e in epoch_entries), 1)
        row["Peak Mem (GB)"] = round(max(e.get("peak_mem_mb", 0) for e in epoch_entries) / 1024, 2)
        row["Time/Epoch (s)"] = round(sum(e["epoch_time_s"] for e in epoch_entries) / len(epoch_entries), 1)
        row["Throughput (tok/s)"] = round(sum(e.get("throughput_toks", 0) for e in epoch_entries) / len(epoch_entries))
    else:
        row.update({"Train Loss": "-", "Val Loss": "-", "Peak Mem (MB)": "-", 
                     "Peak Mem (GB)": "-", "Time/Epoch (s)": "-", "Throughput (tok/s)": "-"})
    
    report_rows.append(row)

df = pd.DataFrame(report_rows).sort_values("Val PPL" if "Val PPL" in report_rows[0] else report_rows[0].keys())
df.to_csv("full_report_table.csv", index=False)

print("=" * 120)
print("FULL REPORT TABLE (saved to full_report_table.csv)")
print("=" * 120)
print(df.to_string(index=False))


print("\n\n" + "=" * 80)
print("EXTRAPOLATION TEST: Train on L=512, Evaluate on L=512/1024/2048")
print("=" * 80)

# Check if extrapolation_results.csv exists
if os.path.exists("extrapolation_results.csv"):
    extrap_df = pd.read_csv("extrapolation_results.csv")
    print(extrap_df.to_string(index=False))
else:
    print("extrapolation_results.csv not found. Run the extrapolation test.")


print("\n\n" + "=" * 80)
print("ATTENTION BENCHMARKS: Throughput & Memory across context lengths")
print("=" * 80)

if os.path.exists("attention_benchmarks.csv"):
    bench_df = pd.read_csv("attention_benchmarks.csv")
    print(bench_df.to_string(index=False))
else:
    print("attention_benchmarks.csv not found. Run the attention benchmarks.")

FULL REPORT TABLE (saved to full_report_table.csv)
         Experiment      Attention    Pos Enc           Conv  Ctx Len  Params  Test PPL  Val PPL  Test Loss Train Loss Val Loss Peak Mem (MB) Peak Mem (GB) Time/Epoch (s) Throughput (tok/s)
        extrap_rope       standard       rope           none      512 4668928    131.83   131.39     4.8815          -        -             -             -              -                  -
       extrap_alibi       standard      alibi           none      512 4668928    135.28   135.99     4.9074          -        -             -             -              -                  -
    extrap_relative       standard   relative           none      512 4669956    172.74   175.42     5.1518          -        -             -             -              -                  -
            pe_rope       standard       rope           none     1024 4668928    175.85   177.82     5.1696          -        -             -             -              -                  -

In [18]:
import os
from pathlib import Path

log_dir = Path("logs")
print("Log files found:")
for f in sorted(log_dir.glob("*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")

print("\n--- Sample log entries from baseline ---")
log_file = log_dir / "baseline_log.jsonl"
if log_file.exists():
    with open(log_file) as f:
        for i, line in enumerate(f):
            if i < 5:
                print(line.strip())

Log files found:
  attn_gqa.jsonl (5541 bytes)
  attn_gqa_summary.json (262 bytes)
  attn_linear.jsonl (7369 bytes)
  attn_linear_summary.json (269 bytes)
  attn_sliding_window.jsonl (5549 bytes)
  attn_sliding_window_summary.json (286 bytes)
  baseline.jsonl (5529 bytes)
  baseline_summary.json (268 bytes)
  conv_before.jsonl (5492 bytes)
  conv_before_summary.json (280 bytes)
  conv_gated_ffn.jsonl (5525 bytes)
  conv_gated_ffn_summary.json (287 bytes)
  conv_interleaved.jsonl (5508 bytes)
  conv_interleaved_summary.json (286 bytes)
  extrap_alibi.jsonl (7268 bytes)
  extrap_alibi_summary.json (267 bytes)
  extrap_relative.jsonl (7385 bytes)
  extrap_relative_summary.json (272 bytes)
  extrap_rope.jsonl (7417 bytes)
  extrap_rope_summary.json (265 bytes)
  extrap_sinusoidal.jsonl (7124 bytes)
  extrap_sinusoidal_summary.json (276 bytes)
  pe_alibi.jsonl (5458 bytes)
  pe_alibi_summary.json (265 bytes)
  pe_relative.jsonl (5536 bytes)
  pe_relative_summary.json (269 bytes)
  pe_rope.j

In [19]:
import json
import pandas as pd
from pathlib import Path

log_dir = Path("logs")

with open(log_dir / "baseline.jsonl") as f:
    for i, line in enumerate(f):
        if i < 5:
            print(json.loads(line))

print("\n\n")

report_rows = []

for summary_file in sorted(log_dir.glob("*_summary.json")):
    exp_name = summary_file.stem.replace("_summary", "")
    
    with open(summary_file) as f:
        summary = json.load(f)
    
    log_file = log_dir / f"{exp_name}.jsonl"
    epoch_entries = []
    if log_file.exists():
        with open(log_file) as f:
            for line in f:
                entry = json.loads(line)
                if "epoch_time_s" in entry:
                    epoch_entries.append(entry)
    
    row = {
        "Experiment": exp_name,
        "Attention": summary.get("attention_type", ""),
        "Pos Enc": summary.get("pos_enc_type", ""),
        "Conv": summary.get("conv_hybrid", "none") or "none",
        "Ctx Len": summary.get("context_len", ""),
        "Params": summary.get("n_params", ""),
        "Train Loss": round(epoch_entries[-1]["train_loss"], 4) if epoch_entries else "-",
        "Val Loss": round(epoch_entries[-1]["val_loss"], 4) if epoch_entries else "-",
        "Val PPL": round(summary.get("best_val_ppl", 0), 2),
        "Test PPL": round(summary.get("test_ppl", 0), 2),
        "Peak Mem (MB)": round(max(e.get("peak_mem_mb", 0) for e in epoch_entries), 1) if epoch_entries else "-",
        "Peak Mem (GB)": round(max(e.get("peak_mem_mb", 0) for e in epoch_entries) / 1024, 2) if epoch_entries else "-",
        "Time/Epoch (s)": round(sum(e["epoch_time_s"] for e in epoch_entries) / len(epoch_entries), 1) if epoch_entries else "-",
        "Throughput (tok/s)": round(sum(e.get("throughput_toks", 0) for e in epoch_entries) / len(epoch_entries)) if epoch_entries else "-",
    }
    report_rows.append(row)

df = pd.DataFrame(report_rows).sort_values("Val PPL")
df.to_csv("full_report_table.csv", index=False)

print("=" * 140)
print("FULL REPORT TABLE (saved to full_report_table.csv)")
print("=" * 140)
print(df.to_string(index=False))

{'epoch': 1, 'train_loss': 9.035973414657374, 'train_ppl': 8399.88601131156, 'val_loss': 8.515606936286478, 'val_ppl': 4992.075011539742, 'throughput_toks': 52934.63731171263, 'epoch_time_s': 49.88975336599992, 'peak_mem_mb': 11422.4326171875}
{'step': 100, 'epoch': 2, 'train_loss': 8.398719787597656, 'train_ppl': 4441.377200386988, 'lr': 5.9999999999999995e-05, 'peak_mem_mb': 11422.4326171875}
{'epoch': 2, 'train_loss': 7.973331151335866, 'train_ppl': 2902.509978126636, 'val_loss': 7.452225124134737, 'val_ppl': 1723.694315202632, 'throughput_toks': 52011.81904177988, 'epoch_time_s': 50.77492094399986, 'peak_mem_mb': 11422.4326171875}
{'step': 200, 'epoch': 3, 'train_loss': 7.350259266401592, 'train_ppl': 1556.6000496186555, 'lr': 0.00011999999999999999, 'peak_mem_mb': 11422.4326171875}
{'epoch': 3, 'train_loss': 7.305282794490902, 'train_ppl': 1488.1407403530652, 'val_loss': 7.246826788958381, 'val_ppl': 1403.6437161527183, 'throughput_toks': 49006.28617882642, 'epoch_time_s': 53.8889

In [1]:
!pip install -q datasets tokenizers tqdm matplotlib pandas
!cp -r /kaggle/input/models/yushkushwaha6753/saidl/tensorflow2/default/1/* /kaggle/working/
import os, sys, time, json
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

import torch
import pandas as pd
from config import ModelConfig, TrainConfig
from train import train
from model import build_model
from utils import get_device

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [2]:
device = get_device()
experiments = [
    ("linear_rope_baseline",    "linear", "rope", None),
    ("linear_rope_conv_before", "linear", "rope", "conv_before"),
    ("linear_rope_interleaved", "linear", "rope", "interleaved"),
    ("linear_rope_gated_conv",  "linear", "rope", "gated_conv_ffn"),
]

summaries = []
for name, attn, pe, conv in experiments:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    cfg = ModelConfig(attention_type=attn, pos_enc_type=pe, conv_hybrid=conv)
    tcfg = TrainConfig(context_len=1024, batch_size=16, epochs=15)  # ← was 32, now 16
    model, summary = train(cfg, tcfg, experiment_name=name)
    summaries.append(summary)
    del model; torch.cuda.empty_cache()


  linear_rope_baseline

Experiment: linear_rope_baseline
Device: cuda



README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training BPE tokenizer...



Tokenizing splits...
  Train tokens: 2,644,133
  Val tokens:   279,575
  Test tokens:  320,220
  Train chunks: 2579
  Val chunks:   272
  Test chunks:  312
Model: linear attn | rope PE | conv=None
  Parameters: 4,668,928
  Step    100 | Loss 8.8691 | PPL 7108.87 | LR 6.00e-05 | Mem 8990MB

Epoch 1/15 | Train Loss 8.4922 PPL 4876.53 | Val Loss 7.4702 PPL 1754.90 | Throughput 30469 tok/s | Time 86.7s

  ✓ Saved best checkpoint (PPL 1754.90)
  Step    200 | Loss 7.3480 | PPL 1553.08 | LR 1.20e-04 | Mem 8990MB
  Step    300 | Loss 7.0930 | PPL 1203.54 | LR 1.80e-04 | Mem 8990MB

Epoch 2/15 | Train Loss 7.0329 PPL 1133.27 | Val Loss 6.5794 PPL 720.13 | Throughput 29932 tok/s | Time 88.2s

  ✓ Saved best checkpoint (PPL 720.13)
  Step    400 | Loss 6.4311 | PPL 620.83 | LR 2.40e-04 | Mem 8990MB

Epoch 3/15 | Train Loss 6.2897 PPL 538.97 | Val Loss 6.0469 PPL 422.81 | Throughput 29388 tok/s | Time 89.9s

  ✓ Saved best checkpoint (PPL 422.81)
  Step    500 | Loss 

In [3]:
@torch.no_grad()
def benchmark(attn, pe, conv, seq_len=1024, batch_size=8, n_runs=20):
    cfg = ModelConfig(attention_type=attn, pos_enc_type=pe, conv_hybrid=conv)
    model = build_model(cfg).to(device).eval()
    x = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)
    
    for _ in range(5):
        _ = model(x)
    torch.cuda.synchronize()
    
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = model(x)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    
    total_tokens = batch_size * seq_len * n_runs
    throughput = total_tokens / elapsed
    latency_ms = (elapsed / n_runs) * 1000
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)
    params = model.count_parameters()
    
    del model; torch.cuda.empty_cache()
    return {
        "params": params,
        "throughput_tok_s": round(throughput),
        "latency_ms": round(latency_ms, 1),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

bench_results = []
for name, attn, pe, conv in experiments:
    print(f"Benchmarking {name}...")
    b = benchmark(attn, pe, conv)
    b["name"] = name
    bench_results.append(b)

Benchmarking linear_rope_baseline...
Model: linear attn | rope PE | conv=None
  Parameters: 4,668,928
Benchmarking linear_rope_conv_before...
Model: linear attn | rope PE | conv=conv_before
  Parameters: 5,982,720
Benchmarking linear_rope_interleaved...
Model: linear attn | rope PE | conv=interleaved
  Parameters: 6,238,720
Benchmarking linear_rope_gated_conv...
Model: linear attn | rope PE | conv=gated_conv_ffn
  Parameters: 9,391,616


In [4]:
rows = []
for s, b in zip(summaries, bench_results):
    rows.append({
        "Architecture": s["experiment"],
        "Conv Hybrid": s.get("conv_hybrid") or "none",
        "Best Val PPL ↓": f"{s['best_val_ppl']:.2f}",
        "Test PPL ↓": f"{s['test_ppl']:.2f}",
        "Params": f"{b['params']:,}",
        "Throughput (tok/s) ↑": f"{b['throughput_tok_s']:,}",
        "Latency (ms) ↓": b["latency_ms"],
        "Peak Mem (MB) ↓": b["peak_mem_mb"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv("part4_perplexity_vs_compute.csv", index=False)
print("\nSaved to part4_perplexity_vs_compute.csv")

           Architecture    Conv Hybrid Best Val PPL ↓ Test PPL ↓    Params Throughput (tok/s) ↑  Latency (ms) ↓  Peak Mem (MB) ↓
   linear_rope_baseline           none         188.27     186.79 4,668,928              117,843            69.5           1158.3
linear_rope_conv_before    conv_before         130.80     130.64 5,982,720               94,820            86.4           1172.2
linear_rope_interleaved    interleaved         133.61     133.55 6,238,720              132,976            61.6           1164.3
 linear_rope_gated_conv gated_conv_ffn         128.94     129.38 9,391,616               63,736           128.5           1176.3

Saved to part4_perplexity_vs_compute.csv
